# Tutorial: CryptoHFTData Apex Backtest Workflow

Audience:
- Apex users who want to prepare CryptoHFTData historical files and run a small backtest against the prepared data.

Prerequisites:
- Apex has been built or can be built locally.
- Apex reference data has been generated with `python/generate-refdata.sh`.
- Python dependencies from `python/requirements-cryptohftdata.txt` are installed.
- `CRYPTOHFTDATA_API_KEY` is set, or you intentionally use the anonymous free tier.

Learning goals:
- Prepare CryptoHFTData `trades` and `orderbook` files into Apex `tickbin1` data.
- Inspect the preparation manifest and a few generated records.
- Run the `apex-example-cryptohftdata-backtest` market-making example on the prepared range.


## Outline

1. Configure the venue, symbol, and Apex-style half-open time range.
2. Run the CryptoHFTData converter.
3. Inspect the manifest and prepared `tickbin1` files.
4. Build or locate the C++ backtest example.
5. Run a simple market-making backtest.
6. Try a small extension.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "CMakeLists.txt").exists():
    REPO_ROOT = REPO_ROOT.parent

APEX_HOME = Path(os.environ.get("APEX_HOME", Path.home() / "apex")).expanduser()
TICKDATA_DIR = APEX_HOME / "data" / "tickdata"

VENUE = os.environ.get("APEX_CHD_VENUE", "binance_usdfut")
SYMBOL = os.environ.get("APEX_CHD_SYMBOL", "BTCUSDT")
START = os.environ.get("APEX_CHD_FROM", "2025-08-01T00:00:00")
UPTO = os.environ.get("APEX_CHD_UPTO", "2025-08-01T01:00:00")
STREAMS = "aggtrades,l1"

# Keep the notebook safe to open. Set this to True, or export
# APEX_CHD_RUN_PREPARE=1, when you want the notebook to download data.
RUN_PREPARE = os.environ.get("APEX_CHD_RUN_PREPARE") == "1"

print("repo:", REPO_ROOT)
print("APEX_HOME:", APEX_HOME)
print("range:", f"[{START}, {UPTO})")
print("dataset:", VENUE, SYMBOL, STREAMS)
print("run prepare:", RUN_PREPARE)


## 1. Preflight

This cell checks that the converter and optional Python dependencies are visible. The API key is never printed.


In [ ]:
converter = REPO_ROOT / "python" / "prepare_cryptohftdata.py"
requirements = REPO_ROOT / "python" / "requirements-cryptohftdata.txt"

print("converter exists:", converter.exists(), converter)
print("requirements file:", requirements)
print("has API key:", bool(os.environ.get("CRYPTOHFTDATA_API_KEY")))

missing = []
for package in ("pyarrow", "zstandard"):
    try:
        __import__(package)
    except ImportError:
        missing.append(package)

if missing:
    print("install missing packages with:")
    print(f"  {sys.executable} -m pip install -r {requirements}")
else:
    print("parquet dependencies are installed")


## 2. Prepare CryptoHFTData for Apex

The converter uses Apex-style half-open time semantics: `[from, upto)`. Raw hourly CryptoHFTData files are deleted after successful conversion unless `--keep-raw` is supplied.


In [ ]:
prepare_cmd = [
    sys.executable,
    str(converter),
    "--venues", VENUE,
    "--symbols", SYMBOL,
    "--streams", STREAMS,
    "--from", START,
    "--upto", UPTO,
    "--jobs", "4",
]

print(" ".join(prepare_cmd))

prepare_result = None
if RUN_PREPARE:
    env = os.environ.copy()
    env["PYTHONPATH"] = str(REPO_ROOT / "python") + os.pathsep + env.get("PYTHONPATH", "")
    completed = subprocess.run(
        prepare_cmd,
        cwd=REPO_ROOT,
        env=env,
        text=True,
        capture_output=True,
        check=True,
    )
    print(completed.stdout)
    prepare_result = json.loads(completed.stdout)
else:
    print("Skipped live preparation. Set APEX_CHD_RUN_PREPARE=1 to execute this cell.")


## 3. Inspect the Manifest

After a live preparation run, this cell loads the manifest. If preparation was skipped, it falls back to the most recent manifest in the Apex tickdata directory.


In [ ]:
manifest_path = None
if prepare_result:
    manifest_path = Path(prepare_result["manifest"])
else:
    manifest_dir = TICKDATA_DIR / "cryptohftdata-manifests"
    manifests = sorted(manifest_dir.glob("prepare-*.json")) if manifest_dir.exists() else []
    manifest_path = manifests[-1] if manifests else None

if manifest_path is None:
    print("No manifest found yet. Run the preparation cell first.")
else:
    manifest = json.loads(manifest_path.read_text())
    print("manifest:", manifest_path)
    print("raw deleted after success:", manifest["raw_deleted_after_success"])
    print("outputs:")
    for item in manifest["outputs"]:
        print(
            f"  {item['venue']} {item['symbol']} {item['stream']} {item['day']} "
            f"emitted={item['emitted']} warnings={len(item['warnings'])}"
        )


## 4. Inspect Prepared `tickbin1` Records

This uses the Python tickbin helper to read a few records from the generated Apex files without starting the C++ engine.


In [ ]:
sys.path.insert(0, str(REPO_ROOT / "python"))
from apex.cryptohftdata.tickbin import iter_records

if manifest_path is None:
    print("No manifest available.")
else:
    for item in manifest["outputs"]:
        path = Path(item["path"])
        stream = item["stream"]
        print("\n", stream, path)
        if not path.exists():
            print("  missing output file")
            continue
        for idx, record in enumerate(iter_records(path, stream)):
            print(" ", record)
            if idx >= 2:
                break


## 5. Build or Locate the Backtest Example

The C++ example is `apex-example-cryptohftdata-backtest`. It reads `APEX_CHD_VENUE`, `APEX_CHD_SYMBOL`, `APEX_CHD_FROM`, and `APEX_CHD_UPTO`, so the notebook and executable use the same range.


In [ ]:
BUILD_CANDIDATES = [
    REPO_ROOT / "BUILD" / "debug",
    REPO_ROOT / "BUILD-DEBUG",
]

exe_rel = Path("src/examples/standalone/apex-example-cryptohftdata-backtest")
example_exe = next((root / exe_rel for root in BUILD_CANDIDATES if (root / exe_rel).exists()), None)

if example_exe:
    print("example executable:", example_exe)
else:
    print("example executable not found yet. Build it with one of:")
    print("  cmake --preset debug")
    print("  cmake --build BUILD/debug --target apex-example-cryptohftdata-backtest")
    print("or, for the older build layout:")
    print("  ./scripts/configure_cmake.sh debug")
    print("  cmake --build BUILD-DEBUG --target apex-example-cryptohftdata-backtest")


## 6. Run the Market-Making Backtest

This runs the standalone Apex backtest over the same half-open range. It expects both `aggtrades` and `l1` files to exist under `$APEX_HOME/data/tickdata/tickbin1`.


In [ ]:
RUN_BACKTEST = os.environ.get("APEX_CHD_RUN_BACKTEST") == "1"

if not example_exe:
    print("No executable found. Build the example first.")
elif not RUN_BACKTEST:
    print("Skipped backtest. Set APEX_CHD_RUN_BACKTEST=1 to execute it.")
    print("Command:", example_exe)
else:
    env = os.environ.copy()
    env.update({
        "APEX_CHD_VENUE": VENUE,
        "APEX_CHD_SYMBOL": SYMBOL,
        "APEX_CHD_FROM": START,
        "APEX_CHD_UPTO": UPTO,
    })
    completed = subprocess.run(
        [str(example_exe)],
        cwd=REPO_ROOT,
        env=env,
        text=True,
        capture_output=True,
        check=False,
    )
    print("return code:", completed.returncode)
    print(completed.stdout[-4000:])
    if completed.stderr:
        print(completed.stderr[-4000:])


## Exercise

Change `VENUE`, `SYMBOL`, `START`, and `UPTO` to prepare a short Bybit range. Keep the range small at first, such as one hour, then compare the manifest counts with the Binance run.

Answer scaffold:
- Set `APEX_CHD_VENUE=bybit`.
- Pick a Bybit symbol that appears in CryptoHFTData symbol discovery, such as `BTCUSDT`.
- Re-run the preparation and manifest cells.
- Re-run the backtest with the same environment variables.


## Common Pitfalls

- If the backtest says no tick files were found, check that the manifest output path matches `$APEX_HOME/data/tickdata/tickbin1/{venue}/{stream}/YYYY/MM/DD/{symbol}.bin`.
- If symbol validation fails, query CryptoHFTData symbols for the venue/data type and use the native symbol exactly.
- If Apex cannot find the instrument, regenerate reference data with `python/generate-refdata.sh` and confirm the venue is one Apex already supports.
- If files already exist, rerun the converter with `--overwrite` only when replacing that prepared data is intentional.
